# Conformalized quantile regression (CQR)

Examples for `uncertainty.ConformalizedQuantileRegressor` and `cross_validate_conformal_quantile`,
using synthetic data (no project data required).

Run from the repo root with `PYTHONPATH=src`, or install the package in editable mode.

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from uncertainty import ConformalizedQuantileRegressor, cross_validate_conformal_quantile

## 1. Explicit calibration (`fit(X, y, X_cal, y_cal)`)

Fit three quantile models on `(X, y)`; compute conformal offset on `(X_cal, y_cal)`.
Point predictions use the **median**; use `predict_interval` for calibrated bounds.

In [ ]:
rng = np.random.default_rng(0)
n = 400
X = rng.standard_normal((n, 5))
y = 0.7 * X[:, 0] - 0.3 * X[:, 2] + 0.15 * rng.standard_normal(n)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)
# Hold out part of *training* for conformal calibration (not used to fit quantiles)
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, random_state=1
)

base = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="quantile",
                quantile=0.5,
                max_iter=150,
                max_depth=4,
                learning_rate=0.08,
                random_state=0,
            ),
        ),
    ]
)

cqr = ConformalizedQuantileRegressor(estimator=base, alpha=0.1)
cqr.fit(X_fit, y_fit, X_cal, y_cal)

y_pred = cqr.predict(X_test)
interval = cqr.predict_interval(X_test)
lower, upper = interval[:, 0], interval[:, 1]

coverage = np.mean((y_test >= lower) & (y_test <= upper))
print(f"Empirical test coverage: {coverage:.3f}  (nominal ~{1 - cqr.alpha:.0%})")
print(f"Mean interval width: {np.mean(upper - lower):.4f}")
print(f"conformal_offset_: {cqr.conformal_offset_:.4f}")

## 2. Same outer CV as the benchmark + nested calibration

`cross_validate_conformal_quantile` uses your `cv` splitter (e.g. `KFold` with the same
`n_splits`, `shuffle`, and `random_state` as `BaselineCVConfig`). Inside each training fold it
splits again into fit vs calibration, then scores **median** predictions on the test fold.

This matches how `run_baseline_cqr_cv` in `baseline.py` evaluates CQR alongside `run_baseline_cv`.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)

base2 = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="quantile",
                quantile=0.5,
                max_iter=120,
                max_depth=4,
                random_state=0,
            ),
        ),
    ]
)
cqr_cv = ConformalizedQuantileRegressor(estimator=base2, alpha=0.1)

out = cross_validate_conformal_quantile(
    cqr_cv,
    X,
    y,
    cv,
    calibration_fraction=0.2,
    random_state=0,
    n_jobs=1,
)
print("Per-fold RMSE:", out["test_rmse"].round(4))
print("Mean RMSE:", out["test_rmse"].mean().round(4), "±", out["test_rmse"].std().round(4))
print("Mean R²:", out["test_r2"].mean().round(4))

## 3. Full training grid (optional)

With real molecules and descriptors, the project entrypoint mirrors `score_data.py`:

```bash
cd openadnet && PYTHONPATH=src python src/score_data_cqr.py
```

That calls `baseline.run_baseline_cqr_cv` with the same `BaselineCVConfig` / `KFold` as
`run_baseline_cv`, and writes `outputs/baseline_cqr_cv_results.csv`.

In [ ]:
# Uncomment when running inside the repo with PYTHONPATH=src and data available:
# from pathlib import Path
# import sys
# sys.path.insert(0, str(Path("..") / "src"))
#
# from rdkit import Chem
# from baseline import BaselineCVConfig, run_baseline_cqr_cv
# from load_data import train
#
# mols = list(train["SMILES"].apply(Chem.MolFromSmiles))
# cfg = BaselineCVConfig(n_splits=5, cv_random_state=0, model_random_state=0)
# results = run_baseline_cqr_cv(train, mols, config=cfg, descriptor_names=["morgan_r2_bits_2048"])
# results.head()